## Targeted Submission Tracks
This submission is specifically engineered and optimized to compete in the following categories:
1. **The Edge AI Prize ($5,000)**: Showcasing a 100% locally hosted, fully functional MedGemma AI-powered agentic workflow running natively on a resource-constrained edge device (Jetson Orin Nano 8GB).

2. **Main Track ($75,000)**: Providing a highly impactful, human-centered solution for medical triage in resource-scarce and internet-deprived environments.

## **AI-Tool Use Disclosure** 

  The project's code, code generation, architectural design, and debugging process involve leveraging and using AI tools to assist with completion. Tools used are ChatGPT 5.2 Extended Thinking under ChatGPT Pro subscription and Google Gemini 3 Pro via Gemini Advanced.

## **Executive Summary** 

   This project is devised from the three pain points faced by modern medical AI applications. Currently, medical AI applications are dependent on the internet, face potential privacy concerns, and experience resource scarcity. The reality is that modern AI medical applications are highly reliant on cloud-based AI model deployment. Despite its high performance, these cloud-based AIs, even if claimed to be HIPAA and GDPR compliant, still require documents and data to be sent to the cloud and processed remotely. Not only that, but the cloud-based nature also limited the use of AI in emergency and disaster scenarios, as well as in the countryside or regions with limited or unreliable internet connections, rendering such use impossible as accessibility becomes increasingly difficult. 
    
   To solve this, we propose a local deployment on a commercially available edge AI device. By deploying it locally on a commercially available edge AI device, we can not only resolve the issue of privacy, but we can also resolve the issue of resource scarcity and the accessibility to medical-grade AI regardless of its parameters. The project is using a 4-bit-quantized Medgemma-1.5-4B model, deployed on a Jetson Orin Nano Super Developer Kit, connected to n8n, which is open source. Both are locally hosted on the said Jetson Orin Nano Super Developer Kit, and the operation of the entire AI agentic workflow itself does not involve an external internet connection. To avoid potential issues such as out-of-memory (OOM), we have added an NVMe SSD which is 500G in size to the Jetson Orin Nano edge AI device and setup a swap file with the size of 22G resolve the issue. 

## **Problem Statement**

   Currently, the medical fields are facing the following hurdles when attempting deploying AI. 
   
   The overreliance of cloud computing and API renders the AI deployment becomes impossible to use in the case of emergency and disaster, because the internet connection will become unreliable or sometimes even unreacheable in these circumstances. Moreover, uploading medical data of patients, even if onto an HIPAA-Compliant provider, can still encounter legal issues when a data breach occured. To resolve these issues associated with the cloud-hosted AI and API-based AI services, we devised a local host solution that is deployable on an AI edge device--in this case it is the Jetson Orin Nano Super Developer Kit--so that the data can be processed locally without leaving the device environment. 
   
   The fact that a lack of primary triage resources in the emergency room or solo practice makes the workload and the number of patients cannot have a fast processing. Moreover, the cognitive workload involved in analyzing the quality of the image and analyzing the image provided can take away precious time and energy in a resource-lacking or a solo-practice environment. We devised a solution by building the AI Agent to offload pressure on the limited resources by providing automation for the primary triage process that is powered by a MedGemma model. The report will be presented in the format of JSON.

Note for Judges: This Kaggle Notebook serves as the official Code Repository for the submission, containing both the training/quantization code and the edge inference logic (n8n workflow & Python webhook trigger script) required for full reproducibility.

## **Solution Architecture**

   Our solution is an AI agent composed of an n8n workflow and a 4-bit quantized MedGemma-1.5-4B, with both hosted on a Jetson Orin Nano Super Developer Kit, an edge AI computing device. 
   
   For hardware, we choose the Jetson Orin Nano Super Developer Kit. It is constrained by 8GB of RAM, but we set up a 22GB swap file on the attached 500G NVMe SSD included with the SDK. All hardware is available on Amazon, making it easy to acquire, low-cost, and easily scalable once the setup process is complete.
   
   For the inference engine, we have chosen Ollama to act as an inference engine. The choice is for providing a reliable port for efficient locally hosted LLM service when paired with the n8n.
   
   For orchestration, we choose to locally host n8n rather than write Python code. We do not host both the LLM and n8n in Docker; we installed them locally, with n8n managed by PM2. We chose n8n also because it provides a visualized, expandable workflow, simplifying maintenance and making it easier for operators to use.

## **Technical Feasibility & Edge Optimization**

   We start the process of optimization and enforcing feasibility by quantizing the MedGemma-1.5-4B model. The quantization of the model is at 4-bit quantization, and the process is shown in the "Quantization" section. 
   Before the "Quantization" section, we also have the performance benchmarks to answer the "how" part regarding how the 4-bit quantized MedGemma-1.5-4B model runs on the Jetson Orin Nano with 8G of RAM and 500G of SSD attached.

### Edge Performance Metrics (Jetson Orin Nano 8GB)
| Metric | Value | Optimization Strategy |
| :--- | :--- | :--- |
| **RAM Usage** | ~5.0 GB (n8n + Model) | **28GB NVMe Swap** + `vm.swappiness=25` |
| **Inference Time** | 4-8 seconds | 4-bit Quantization (GGUF) |
| **Storage** | 2.5 GB (Model) | 4-bit Quantized,reduced from 9GB (FP16) |
| **Connection Status** | **100% On Edge AI Device** | Local n8n + Ollama |

## **Quantization**

We start by quantizing MedGemma-1.5-4B to prepare for deployment on edge AI devices. The process is outlined in the following code blocks, annotated with markdown.

Computational Requirements for Build (Cloud): Running this notebook to quantize the model requires a Kaggle environment with at least an Nvidia T4 GPU (or equivalent), 16GB+ System RAM, and ~20GB of disk space for intermediate Hugging Face and GGUF files.

**Step 1**: Install Torch and Transformers by Google with other dependencies. 

We locked the Google Transformers at version 4.57.6 because it was the exact version I used when I engaged fine-tuning and quantization in the Kaggle environment. For newer versions released after the date of finetuning and quantization, when using instructions transformers>= 4.50.0, it will install, by default, the newest versions, which, after several attempts, we found break support for multimodal quantization. 

In [ ]:
import sys, torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)

!python -m pip -q install -U pip setuptools wheel

!python -m pip -q install -U --no-cache-dir \
  "transformers==4.57.6" \
  "accelerate>=0.29.0" \
  "peft>=0.10.0" \
  "datasets>=2.18.0" \
  "trl>=0.8.6" \
  "bitsandbytes>=0.43.0" \
  "huggingface_hub>=0.23.0" \
  "tokenizers" \
  "safetensors" \
  "packaging"

import transformers, accelerate, peft, datasets, trl, bitsandbytes as bnb
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)
print("bnb:", bnb.__version__)

**Step 2**: install llama.cpp.

In [ ]:
%%bash
set -e
cd /kaggle/working

rm -rf llama.cpp
git clone --depth 1 https://github.com/ggml-org/llama.cpp
cd llama.cpp

cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j --target llama-quantize

echo "llama-quantize path:"
find build -maxdepth 4 -type f -name "llama-quantize" -print

**Step 3**: Train the MedGemma model.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

import torch
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 0) Sanity
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))

# 1) HF login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HackathonNewFrontier")
login(token=hf_token)

# 2) Dataset (small, then scale)
ds = load_dataset("medalpaca/medical_meadow_wikidoc", split="train")

def to_text(ex):
    return {"text": f"### Instruction:\n{ex['input']}\n\n### Response:\n{ex['output']}"}

ds = ds.map(to_text, remove_columns=ds.column_names)

MODEL_ID = "google/medgemma-1.5-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 512

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

ds_tok = ds.map(tok, batched=True, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3) 4-bit load (T4 friendly; use fp16 compute on T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# stability knobs
model.config.use_cache = False
model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)

out_dir = "outputs"
args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=30,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_steps=10,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok,
    data_collator=collator,
)

trainer.train()
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
print("saved:", out_dir)

Step 4: Export the trained MedGemma model.

In [ ]:
import os, glob, time, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Release RAM to avoid OOM
try:
    del model
    del trainer
    del base
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared. Starting export process...")

MODEL_ID = "google/medgemma-1.5-4b-it"
EXPORT_NAME = "medgemma_edge_hf"

print("Searching for checkpoints...")
max_retries = 10
latest_ckpt = None

for i in range(max_retries):
    ckpts = glob.glob("outputs/checkpoint-*")
    if ckpts:
        latest_ckpt = max(ckpts, key=os.path.getctime)
        print(f"Checkpoints found: {latest_ckpt}")
        break
    
    print(f"Waiting for checkpoints... ({i+1}/{max_retries})")
    time.sleep(5) 

if not latest_ckpt:
    print("WARNING: No checkpoints found! Loading base model only.")
else:
    print(f"Target checkpoint: {latest_ckpt}")

# Load Model
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": "cpu"}, 
    dtype=torch.float16,
    trust_remote_code=True,
)

if latest_ckpt:
    print(f"Loading LoRA adapters from {latest_ckpt}...")
    model = PeftModel.from_pretrained(base, latest_ckpt)
    print("Merging model...")
    model = model.merge_and_unload()
else:
    model = base

# === 存檔 ===
print("Saving merged model...")
model.save_pretrained(EXPORT_NAME)

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tok.save_pretrained(EXPORT_NAME)

print(f"SUCCESS: Model saved to {EXPORT_NAME}")

**Step 5**: Quantize and export the MedGemma 1.5 4B model. The quantization here will be set at 4 bits.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working

python llama.cpp/convert_hf_to_gguf.py medgemma_edge_hf \
  --outfile medgemma_edge_hf.f16.gguf \
  --outtype f16

LLAMA_Q="llama.cpp/build/bin/llama-quantize"
test -f "$LLAMA_Q"

"$LLAMA_Q" medgemma_edge_hf.f16.gguf medgemma_edge_hf.Q4_K_M.gguf q4_k_m

echo "Cleaning up intermediate files to save space..."
rm -f medgemma_edge_hf.f16.gguf       # Delete Intermediate Model Version to save HDD Space
rm -rf medgemma_edge_hf               # Delete Original Model version to save HDD Space

ls -lh medgemma_edge_hf.Q4_K_M.gguf

## **n8n Workflow in JSON code**

   The code block of the n8n workflow is fully pasted below in markdown cell to avoid error:

{
  "name": "A Lightweight MedGemma-based Edge Triage AI Agent",
  "nodes": [
    {
      "parameters": {
        "public": true,
        "options": {
          "title": "Hi there! 👋"
        }
      },
      "type": "@n8n/n8n-nodes-langchain.chatTrigger",
      "typeVersion": 1.4,
      "position": [
        -464,
        -192
      ],
      "id": "c151d196-3846-4e6d-8bd4-52fdca59b2ef",
      "name": "When chat message received",
      "webhookId": "533e4405-d14d-4cfe-b2a4-852af019a187"
    },
    {
      "parameters": {
        "promptType": "define",
        "text": "={{ $json.final_prompt }}",
        "batching": {}
      },
      "type": "@n8n/n8n-nodes-langchain.chainLlm",
      "typeVersion": 1.9,
      "position": [
        -192,
        -400
      ],
      "id": "d0d0b2dc-3ede-4194-a18c-c8fbeb1b5c92",
      "name": "Basic LLM Chain",
      "onError": "continueRegularOutput"
    },
    {
      "parameters": {
        "conditions": {
          "options": {
            "caseSensitive": true,
            "leftValue": "",
            "typeValidation": "strict",
            "version": 3
          },
          "conditions": [
            {
              "id": "d506a8e8-82ad-434b-8c34-0c86e15f0f06",
              "leftValue": "=={{ $json.body.image }}",
              "rightValue": "webhook",
              "operator": {
                "type": "string",
                "operation": "notEmpty",
                "singleValue": true
              }
            }
          ],
          "combinator": "and"
        },
        "options": {}
      },
      "type": "n8n-nodes-base.if",
      "typeVersion": 2.3,
      "position": [
        -48,
        -176
      ],
      "id": "bd3f2f4d-f16b-4f48-8132-eb7109e0386a",
      "name": "If"
    },
    {
      "parameters": {
        "respondWith": "json",
        "responseBody": "={{ $json }}",
        "options": {}
      },
      "type": "n8n-nodes-base.respondToWebhook",
      "typeVersion": 1.5,
      "position": [
        160,
        -192
      ],
      "id": "b8059c54-639c-41a7-8e05-f5b32c3cf494",
      "name": "Respond to Webhook"
    },
    {
      "parameters": {
        "httpMethod": "POST",
        "path": "34d83fa3-52e0-417d-b76c-227944f38e54",
        "responseMode": "responseNode",
        "options": {}
      },
      "type": "n8n-nodes-base.webhook",
      "typeVersion": 2.1,
      "position": [
        -464,
        -400
      ],
      "id": "7e621878-13fb-4d69-bb8a-ce01d6fa5481",
      "name": "Webhook",
      "webhookId": "34d83fa3-52e0-417d-b76c-227944f38e54"
    },
    {
      "parameters": {
        "assignments": {
          "assignments": [
            {
              "id": "2c6f7dc9-d70c-4346-81b1-f8c59f57da18",
              "name": "final_prompt",
              "value": "={{ \n  $json.chatInput \n  || $json.body?.query \n  || $json.query \n  || $json.body?.message \n  || $json.body?.text \n  || '' \n}}",
              "type": "string"
            },
            {
              "id": "2676cbfa-4b72-40ab-8d3e-fecea8e713eb",
              "name": "image",
              "value": "={{ $json.body?.image || '' }}",
              "type": "string"
            },
            {
              "id": "d460c056-77ec-4077-9778-394757ad97a9",
              "name": "source",
              "value": "={{ $json.body ? 'webhook' : 'chat' }}",
              "type": "string"
            },
            {
              "id": "c085d07e-142a-4afc-ba0d-c869339b8091",
              "name": "start_time",
              "value": "={{ $now }}",
              "type": "string"
            }
          ]
        },
        "options": {}
      },
      "type": "n8n-nodes-base.set",
      "typeVersion": 3.4,
      "position": [
        -320,
        -400
      ],
      "id": "979ecd46-cc78-4c91-afdd-386b578ead31",
      "name": "Edit Fields"
    },
    {
      "parameters": {
        "model": "medgemma-edge:latest",
        "options": {
          "temperature": 0.6,
          "topP": 0.9,
          "numCtx": 1500
        }
      },
      "type": "@n8n/n8n-nodes-langchain.lmChatOllama",
      "typeVersion": 1,
      "position": [
        -192,
        -160
      ],
      "id": "4daeb809-668e-46eb-a9bc-ea87c84a4e23",
      "name": "Ollama Chat Model",
      "credentials": {
        "ollamaApi": {
          "id": "jBoS8nmuZriubSx7",
          "name": "Ollama account"
        }
      }
    },
    {
      "parameters": {
        "assignments": {
          "assignments": [
            {
              "id": "e519674b-b050-40c9-af03-f175b3f1cb43",
              "name": "text",
              "value": "={{ $json.text || $json.output || $json.response || '' }}",
              "type": "string"
            }
          ]
        },
        "includeOtherFields": true,
        "options": {}
      },
      "type": "n8n-nodes-base.set",
      "typeVersion": 3.4,
      "position": [
        240,
        -400
      ],
      "id": "f050f95d-ebf1-41d3-a625-204d441cc95e",
      "name": "Edit Fields1"
    },
    {
      "parameters": {
        "jsCode": "// ==========================================\n// 1. 修正延遲計算 (Fix Latency Logic)\n// ==========================================\n// 關鍵修正：不能用 $input，因為 LLM 節點把 start_time 弄丟了。\n// 必須用 $('節點名稱') 直接回溯讀取 LLM 之前的資料。\n// 請確認你的畫布上，LLM 左邊那個節點名字真的叫 'Edit Fields'\nlet startTimeStr;\ntry {\n  startTimeStr = $('Edit Fields').first().json.start_time;\n} catch (e) {\n  startTimeStr = null; \n}\n\n// 如果抓不到開始時間，就用當下時間（避免代碼崩潰，雖然延遲會變 0）\nconst startTime = startTimeStr ? new Date(startTimeStr).getTime() : new Date().getTime();\nconst endTime = new Date().getTime();\nconst latency = endTime - startTime;\n\n// ==========================================\n// 2. 獲取 LLM 原始輸出\n// ==========================================\nconst raw_text = $input.first().json.text || \"\";\n\n// ==========================================\n// 3. 執行分流邏輯 (The Branching Magic)\n// ==========================================\n// 定義正則表達式：只抓取 { ... } 範圍內的 JSON\nconst regex = /\\{[\\s\\S]*\\}/;\nconst match = raw_text.match(regex);\n\nlet final_output = {};\n\nif (match) {\n  // --- 分支 A：Webhook 模式 (成功抓到 JSON) ---\n  try {\n    const parsed_json = JSON.parse(match[0]);\n    // 成功解析！展開所有欄位\n    final_output = {\n      ...parsed_json,\n      is_structured: true\n    };\n  } catch (error) {\n    // --- 分支 B：JSON 格式錯誤 (你要求的保留報錯部分) ---\n    final_output = { \n      error: \"JSON parse failed\", \n      raw_fragment: match[0].substring(0, 100),\n      is_structured: false\n    };\n  }\n} else {\n  // --- 分支 C：Chat 模式 (完全沒有 JSON) ---\n  // 這裡不應該報錯，因為這代表模型正在跟你聊天\n  // 我們把原始文字直接塞進 assessment_result，這樣 Chat 視窗就能顯示了\n  final_output = { \n    assessment_result: raw_text, // 讓聊天內容能被看到\n    warning: \"No JSON structure found (Chat Mode)\",\n    is_structured: false\n  };\n}\n\n// ==========================================\n// 4. 輸出最終結果\n// ==========================================\nreturn {\n  json: {\n    // 把上面的分析結果攤平放出來\n    ...final_output,\n    \n    // 加上技術指標 (Metadata)\n    _metadata: {\n      latency_ms: latency, // 這次這裡會有真實數字了\n      model: \"MedGemma-4B-Quantized\",\n      device: \"NVIDIA Jetson Orin Nano (8GB)\",\n      timestamp: new Date().toISOString()\n    }\n  }\n};"
      },
      "type": "n8n-nodes-base.code",
      "typeVersion": 2,
      "position": [
        96,
        -400
      ],
      "id": "5c579a0c-a696-4b81-92c5-18f5e4b92b0c",
      "name": "Code in JavaScript"
    }
  ],
  "pinData": {},
  "connections": {
    "When chat message received": {
      "main": [
        [
          {
            "node": "Edit Fields",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Basic LLM Chain": {
      "main": [
        [
          {
            "node": "Code in JavaScript",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "If": {
      "main": [
        [
          {
            "node": "Respond to Webhook",
            "type": "main",
            "index": 0
          }
        ],
        []
      ]
    },
    "Webhook": {
      "main": [
        [
          {
            "node": "Edit Fields",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Edit Fields": {
      "main": [
        [
          {
            "node": "Basic LLM Chain",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Ollama Chat Model": {
      "ai_languageModel": [
        [
          {
            "node": "Basic LLM Chain",
            "type": "ai_languageModel",
            "index": 0
          }
        ]
      ]
    },
    "Edit Fields1": {
      "main": [
        [
          {
            "node": "If",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Code in JavaScript": {
      "main": [
        [
          {
            "node": "Edit Fields1",
            "type": "main",
            "index": 0
          }
        ]
      ]
    }
  },
  "active": false,
  "settings": {
    "executionOrder": "v1",
    "availableInMCP": false
  },
  "versionId": "1c6829c2-1dea-4165-8063-adc7ec47cdf7",
  "meta": {
    "templateCredsSetupCompleted": true,
    "instanceId": "d571f823f62d960da4fe4591c42a9c128dc6d53ec9d197ae023a92179309abdd"
  },
  "id": "U0dexKxEEPgVDVaQ",
  "tags": []
}

## How we managed to run it on an Edge AI device (Instructions)

Device Name: Jetson Orin Nano 8G Super Developer Kit

Environment: Linux Distro Ubuntu 22.04 with GUI enabled

Installation method: Terminal instructions

Operation steps:

1. **Install Ollama**: `curl -fsSL https://ollama.com/install.sh | sh`
2. **Load Model**: `ollama create medgemma-edge -f Modelfile` (using the quantized .gguf)
3. **Start Orchestrator**:
   `npm install -g n8n pm2`
   `N8N_SECURE_COOKIE=false pm2 start n8n`

## **Python Code**

   Here is the Python code that triggers the transition of the image into the binary code and send it down the workflow of the n8n.

import os
import json
import base64
import uuid
import time
import requests

WEBHOOK_URL = os.environ.get("N8N_WEBHOOK_URL", "http://192.168.12.164:5678/webhook-test/34d83fa3-52e0-417d-b76c-227944f38e54")

IMAGE_FILENAME = os.environ.get("XRAY_IMAGE", "xray_sample.png")

TIMEOUT_SEC = float(os.environ.get("HTTP_TIMEOUT", "60"))
RETRY = int(os.environ.get("HTTP_RETRY", "2"))
RETRY_BACKOFF_SEC = float(os.environ.get("HTTP_RETRY_BACKOFF", "1.5"))
# =================================================


def get_image_path(filename: str) -> str:
    script_dir = os.path.dirname(os.path.abspath(__file__))
    return os.path.join(script_dir, filename)


def encode_image_b64(image_path: str) -> str:
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found! Please check the path: {image_path}")

    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def post_with_retry(url: str, payload: dict) -> requests.Response:
    last_exc = None
    for attempt in range(RETRY + 1):
        try:
            # json=payload 會自動設定 Content-Type: application/json
            return requests.post(url, json=payload, timeout=TIMEOUT_SEC)
        except Exception as e:
            last_exc = e
            if attempt < RETRY:
                sleep_s = RETRY_BACKOFF_SEC * (attempt + 1)
                print(f"[WARN] POST failed (attempt {attempt+1}/{RETRY+1}): {e}. Retrying in {sleep_s:.1f}s ...")
                time.sleep(sleep_s)
    raise last_exc


def safe_parse_response(resp: requests.Response) -> dict:
    """
    # 盡量把回應解析成 JSON；如果不是 JSON，也會把原文保留下來，
    # 避免你之前遇到的 'Expecting value: line 1 column 1 (char 0)' 直接炸掉。
    """
    result = {
        "status_code": resp.status_code,
        "content_type": resp.headers.get("content-type", ""),
        "text": None,
        "json": None,
    }

    # 一定先把 raw text 留下來（debug 超重要）
    try:
        result["text"] = resp.text
    except Exception:
        result["text"] = "<unable to read resp.text>"

    # 再嘗試 JSON
    try:
        result["json"] = resp.json()
    except Exception:
        result["json"] = None

    return result


def build_payload(img_b64: str) -> dict:
    """
    # 方法 A：影像品質檢查 demo payload
    """
    return {
        "request_id": str(uuid.uuid4()),
        "task": "xray_quality_check",
        "query": (
            "Demo task: Assess ONLY technical/quality aspects of this image and write out the assessment result in natural paragraphs format. "
            "Do NOT interpret medical findings, do NOT diagnose, and do NOT give medical advice, BUT take into account the body section the image shows. "
            "Output JSON."
        ),
        "output_schema": {
            "modality": "string",
            "view_guess": "string",
            "quality_issues": ["string"],
            "confidence": "number",
            "limitations": "string"
        },
        "history": "De-identified demo input. No clinical context. Evaluate image quality only.",
        "image_b64": img_b64
    }


if __name__ == "__main__":
    req_id = str(uuid.uuid4())
    img_path = get_image_path(IMAGE_FILENAME)

    print(f"[{req_id}] Reading image: {img_path}")
    img_b64 = encode_image_b64(img_path)

    payload = build_payload(img_b64)

    print(f"[{req_id}] POST -> {WEBHOOK_URL}")
    print("(Make sure n8n is listening if you're using webhook-test.)")

    resp = post_with_retry(WEBHOOK_URL, payload)
    parsed = safe_parse_response(resp)

    print(f"\n[{req_id}] === RESPONSE SUMMARY ===")
    print(f"Status: {parsed['status_code']}")
    print(f"Content-Type: {parsed['content_type']}")

    if parsed["json"] is not None:
        print("\nJSON:")
        print(json.dumps(parsed["json"], indent=2, ensure_ascii=False))
    else:
        print("\nResponse is NOT JSON. Raw text:")
        print(parsed["text"])


**Note**: The IP address in the Python code is used in the same Wi-Fi network when connected to the Jetson Orin Nano Super Developer Kit.
If you attempt to use the code, click to enter the cell edit mode and copy the code.

## Link to the Video Demonstration:

https://drive.google.com/file/d/17RdZL55l-WIG7Rbzk2RPlc7TfdY7pqSR/view?usp=sharing

**Important**: Please refer to the video demo for visual proof of the hardware setup!


## License & Compliance
* **Submission License**: This project (including code, workflow JSON, and documentation) is licensed under **CC BY 4.0**.
* **Model Terms**: The use of HAI-DEF and MedGemma model MedGemma-1.5-4B adheres to the **HAI-DEF Terms of Use**.

## Acknowledgements & AI Disclosure
This project represents a **human-AI collaboration**.
* **Lead Architect**: Taiyou Lin - System Architecture, Hardware Integration, and Validation.
* **AI Co-Pilots**:
    * **ChatGPT 5.2 (Extended Thinking)** & **Google Gemini 3 Pro**: Assisted in Python scripting, n8n logic generation, cross-verification, and error debugging.
    * **Google Grounding**: Used for real-time fact-checking of competition rules and medical terminologies.

*We believe this project demonstrates how AI-augmented development can empower individual developers to solve complex, real-world problems typically requiring entire engineering teams.*